# 02 — LiDAR Point Cloud Projection
Project 3D LiDAR points into 2D camera image space and visualize depth.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install numpy matplotlib opencv-python -q

In [ ]:
import os, cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm

BASE_PATH  = '/content/drive/MyDrive/Project/Kitti/Training'
IMAGE_PATH = os.path.join(BASE_PATH, 'image_2/training/image_2')
LIDAR_PATH = os.path.join(BASE_PATH, 'velodyne/training/velodyne')
CALIB_PATH = os.path.join(BASE_PATH, 'calib/data_object_calib/training/calib')

def parse_calib(calib_path):
    calib = {}
    with open(calib_path, 'r') as f:
        for line in f.readlines():
            if ':' in line:
                key, value = line.split(':', 1)
                calib[key.strip()] = np.array(
                    [float(x) for x in value.strip().split()])
    return (calib['P2'].reshape(3,4),
            calib['R0_rect'].reshape(3,3),
            calib['Tr_velo_to_cam'].reshape(3,4))

def project_lidar_to_image(points_3d, P2, R0, Tr):
    N        = points_3d.shape[0]
    pts_hom  = np.hstack([points_3d, np.ones((N,1))])
    Tr_full  = np.vstack([Tr, [0,0,0,1]])
    R0_full  = np.eye(4); R0_full[:3,:3] = R0
    pts_cam  = R0_full @ Tr_full @ pts_hom.T
    depth    = pts_cam[2,:]
    valid    = depth > 0
    pts_cam  = pts_cam[:,valid]; depth = depth[valid]
    pts_img  = P2 @ pts_cam
    pts_img  = pts_img / pts_img[2,:]
    pixels   = pts_img[:2,:].T
    return pixels, depth, valid

sample_id = sorted(os.listdir(IMAGE_PATH))[0].replace('.png','')
P2, R0, Tr = parse_calib(f'{CALIB_PATH}/{sample_id}.txt')
img        = cv2.imread(f'{IMAGE_PATH}/{sample_id}.png')
img_rgb    = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
h, w       = img.shape[:2]
pts        = np.fromfile(f'{LIDAR_PATH}/{sample_id}.bin', dtype=np.float32).reshape(-1,4)
pts_xyz    = pts[:,:3]
pixels, depth, valid = project_lidar_to_image(pts_xyz, P2, R0, Tr)

px, py      = pixels[:,0], pixels[:,1]
in_bounds   = (px>=0)&(px<w)&(py>=0)&(py<h)
depth_clip  = np.clip(depth, 0, 50)

print(f'Image: {w}x{h} | LiDAR points: {len(pts_xyz):,} | In image: {in_bounds.sum():,}')
print(f'Depth — min: {depth.min():.1f}m  max: {depth.max():.1f}m  mean: {depth.mean():.1f}m')

In [ ]:
plt.figure(figsize=(16,6))
plt.imshow(img_rgb)
scatter = plt.scatter(px[in_bounds], py[in_bounds],
                      c=depth_clip[in_bounds], cmap='jet',
                      s=2, alpha=0.8, vmin=0, vmax=50)
plt.colorbar(scatter, label='Distance (meters)')
plt.title(f'LiDAR Points Projected onto Camera — Sample {sample_id}')
plt.axis('off'); plt.tight_layout(); plt.show()